In [1]:
pip install pymupdf requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 50.3 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF for PDF processing
import requests
import re


In [4]:
import fitz  # PyMuPDF for PDF processing
import requests
import re

# Groq API details (Replace with actual API key and endpoint)
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
API_KEY = ""

def extract_text_from_pdf(pdf_path):
    """Extracts text from a textual PDF."""
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")

        if text.strip():
            full_text += text + "\n"

    return clean_text(full_text.strip())

def clean_text(text):
    """Cleans extracted text by removing unnecessary characters and whitespace."""
    text = re.sub(r'\s+', ' ', text)  # Remove excessive whitespace
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # Remove special characters (except punctuation)
    return text.strip()

def summarize_text(text, max_tokens=5000):
    """Trims long text and sends to Groq API for summarization."""
    trimmed_text = text[:max_tokens]  # Reduce text size

    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": "llama-3.1-8b-instant",  # Adjust model if needed
        "messages": [
            {"role": "system", "content": "Summarize the following document concisely."},
            {"role": "user", "content": trimmed_text}
        ]
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=payload)

    # Debugging: Print response to check token size
    print("API Response:", response.json())

    return response.json().get("choices", [{}])[0].get("message", {}).get("content", "Summarization failed.")

# Example usage
pdf_path = "HR_Questions.pdf"  # Change this to your PDF file path
text = extract_text_from_pdf(pdf_path)
summary = summarize_text(text)
print("Summary:", summary)


API Response: {'id': 'chatcmpl-533b0530-7ba3-45be-a7c2-fba3d005ac3b', 'object': 'chat.completion', 'created': 1739639515, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'This document provides concise guidance on how to answer common HR interview questions. The questions and corresponding recommended answers are as follows:\n\n1. **How are you today?**\n- Keep your answer short and simple, e.g., "I\'m doing great, thank you. It\'s good to be here."\n\n2. **Tell me something about yourself.**\n- Two examples are provided:\n  - For freshers: "I\'m among the top graduates of my batch. I have a degree in X subject and an MBA in Operations/Digital Marketing/Business Analytics/International Business. I love exploring new domains and am a fast learner."\n  - For experienced professionals: "I\'ve been dedicated to administrative and managerial work for the past few years. I\'ve worn multiple hats like Business Analyst, Team Lead, and Proje